In [1]:
import os
import numpy as np
import xarray as xr
import rioxarray
from dask.distributed import Client, LocalCluster, progress
from dask.diagnostics import ProgressBar
import dask
import xsdba

print(xsdba.__version__)

0.5.0


In [ ]:
# -----------------------------
# Paths to data
# -----------------------------
OBS_PATH = "/home/dafcluster4/Documents/GitHub/TraCE_Sahul/02_data/02_processed/CHELSA/CHELSA_0p05_pr_climatology.nc"
SIM_PATH = "/home/dafcluster4/Documents/GitHub/TraCE_Sahul/02_data/02_processed/TRACE/TraCE_0p05_pr_climatology.nc"
TEST_PATH = "/media/dafcluster4/storage/TraCE_1500_1990CE/1500_1990/out/pr/TraCE_downscaled_1500_1990_concat.nc"

OUTPUT_QDM_TRAINED = "/home/dafcluster4/Documents/GitHub/TraCE_Sahul/python_test/qdm_trained_pr_Sahul.nc"
OUTPUT_ADJUSTED = "/media/dafcluster4/storage/TraCE_1500_1990CE/1500_1990/out/pr/TraCE_22ka_downscaled_pr_1500_1989_qdm.nc"

# chunking: 50 years -> 50 * 12 months
YEARS_PER_CHUNK = 50
MONTHS_PER_CHUNK = YEARS_PER_CHUNK * 12  # = 600

# subset test data to this period
TEST_SUBSET_START = "1500-01-01"
TEST_SUBSET_END = "1989-12-31"

In [ ]:
# -----------------------------
# Dask config (processes avoids GIL issues)
# -----------------------------
# dask.config.set(scheduler="processes")

# Dask local cluster config tuned to 128 cores / 250 GB
N_WORKERS = 16                 # 64 workers × 2 threads = 128 threads
THREADS_PER_WORKER = 4
MEMORY_LIMIT_PER_WORKER = "12.0GB"  # ~16 * 12 = 192 GB total

# Shut down any previous cluster in this notebook session
try:
    client.close()
    cluster.close()
except:
    pass

# Start a new cluster
cluster = LocalCluster(
    n_workers=N_WORKERS,
    threads_per_worker=THREADS_PER_WORKER,
    memory_limit=MEMORY_LIMIT_PER_WORKER,
    processes=True,
    dashboard_address=":43051"
)
client = Client(cluster)

client

In [ ]:
# -----------------------------
# Open small climatologies (obs, sim) with Dask
# -----------------------------
print("Opening climatology files (lazy, with dask chunks)...")
obs = xr.open_dataset(OBS_PATH, decode_times=False, chunks={"time": -1})["pr"]
sim = xr.open_dataset(SIM_PATH, decode_times=False, chunks={"time": -1})["pr"]

# Ensure time coordinates are months Jan-Dec (climatologies)
months_clim = xr.date_range(
    start="1985-01-01", end="1985-12-31", freq="ME", calendar="noleap", use_cftime=True
)
obs = obs.assign_coords(time=months_clim)
sim = sim.assign_coords(time=months_clim)

# Normalize name of lat/lon if needed
if "longitude" in obs.coords: obs = obs.rename({"longitude": "lon"})
if "latitude" in obs.coords: obs = obs.rename({"latitude": "lat"})
if "longitude" in sim.coords: sim = sim.rename({"longitude": "lon"})
if "latitude" in sim.coords: sim = sim.rename({"latitude": "lat"})

print("obs:", obs)
print("sim:", sim)

In [ ]:
# -----------------------------
# Open large simulated dataset with Dask and subset to 1950-1989
# -----------------------------
print("Opening large test dataset (lazy) and subsetting to 1950-01 -> 1989-12 ...")
downTrace = xr.open_dataset(TEST_PATH, decode_times=False, chunks={"time": MONTHS_PER_CHUNK})["pr"]

# assign full time coords then subset
months_full = xr.date_range(
    start="1500-01-01", end="1989-12-31", freq="ME", calendar="noleap", use_cftime=True
)
downTrace = downTrace.assign_coords(time=months_full)

# subset to requested window
downTrace = downTrace.sel(time=slice(TEST_SUBSET_START, TEST_SUBSET_END))
print("downTrace (subset):", downTrace)
print("downTrace size (months):", downTrace.sizes.get("time"))

In [ ]:
# -----------------------------
# Assign CRS (rioxarray) - ensure arrays can be reprojected if needed
# -----------------------------
print("Writing CRS (EPSG:4326) to DataArrays (lazy metadata only)...")
obs = obs.rio.write_crs("EPSG:4326")
sim = sim.rio.write_crs("EPSG:4326")
downTrace = downTrace.rio.write_crs("EPSG:4326")

# -----------------------------
# Reproject obs/sim to grid of downTrace
# -----------------------------
# rioxarray needs x/y names for reproject; create x/y copies for the operation
print("Preparing names for reprojection (rioxarray expects x/y)...")
obs_xy = obs.rename({"lat": "y", "lon": "x"})
sim_xy = sim.rename({"lat": "y", "lon": "x"})
downTrace_xy = downTrace.rename({"lat": "y", "lon": "x"})

print("Reprojecting obs and sim to match downTrace grid...")
# reproject_match returns a lazily reprojected DataArray (but may trigger some work)
obs_xy = obs_xy.rio.reproject_match(downTrace_xy)
sim_xy = sim_xy.rio.reproject_match(downTrace_xy)

# Rename back to lat/lon
obs = obs_xy.rename({"y": "lat", "x": "lon"})
sim = sim_xy.rename({"y": "lat", "x": "lon"})
downTrace = downTrace_xy.rename({"y": "lat", "x": "lon"})

In [ ]:
# -----------------------------
# Rechunk datasets to 50-year time chunks and reasonable spatial chunks
# -----------------------------
print("Calculating dynamic lat/lon chunk sizes...")
# suggested splits: approx 5 splits along each spatial axis (tunable)
n_spatial_splits = 5
lat_len = downTrace.sizes["lat"]
lon_len = downTrace.sizes["lon"]
lat_chunk = max(1, lat_len // n_spatial_splits)
lon_chunk = max(1, lon_len // n_spatial_splits)

print(f"Using chunk sizes -> time: {MONTHS_PER_CHUNK}, lat: {lat_chunk}, lon: {lon_chunk}")

obs = obs.chunk({"time": -1, "lat": lat_chunk, "lon": lon_chunk})
sim = sim.chunk({"time": -1, "lat": lat_chunk, "lon": lon_chunk})
downTrace = downTrace.chunk({"time": MONTHS_PER_CHUNK, "lat": lat_chunk, "lon": lon_chunk})

# Print chunk info for sanity
print("Chunks (obs):", obs.chunks)
print("Chunks (sim):", sim.chunks)
print("Chunks (downTrace):", downTrace.chunks)

In [ ]:
# -----------------------------
# Utility: bbox printer (no heavy compute)
# -----------------------------
def print_bbox(da, name="DataArray"):
    # we call .values on coords only (small)
    lat_vals = xr.DataArray(da["lat"]).values
    lon_vals = xr.DataArray(da["lon"]).values
    lat_min, lat_max = float(np.nanmin(lat_vals)), float(np.nanmax(lat_vals))
    lon_min, lon_max = float(np.nanmin(lon_vals)), float(np.nanmax(lon_vals))
    print(f"{name} bounding box:")
    print(f"  lat: {lat_min} → {lat_max}")
    print(f"  lon: {lon_min} → {lon_max}")
    print()

print_bbox(obs, "obs")
print_bbox(sim, "sim")
print_bbox(downTrace, "downTrace (subset)")

In [ ]:
# -----------------------------
# Train (or load existing) QDM (xsdba) using climatologies
# -----------------------------

if os.path.exists(OUTPUT_QDM_TRAINED):
    print(f"Found existing trained QDM: {OUTPUT_QDM_TRAINED}")
    ds = xr.open_dataset(OUTPUT_QDM_TRAINED).load()
    qdm = xsdba.adjustment.QuantileDeltaMapping.from_dataset(ds)
    ds.close()  
    print(qdm.ds)  
else:
    print("Training QDM...")
    qdm = xsdba.adjustment.QuantileDeltaMapping.train(
        obs, sim, nquantiles=101, kind="*", group="time.month"
    )

    print("QDM trained; dataset summary:")
    print(qdm.ds)

    # Save trained QDM to NetCDF
    print(f"Saving trained QDM to {OUTPUT_QDM_TRAINED} ...")
    with ProgressBar():
        qdm.ds.to_netcdf(OUTPUT_QDM_TRAINED, compute=True)
    print("QDM saved.")

# print("Training QDM...")
# qdm = adjustment.QuantileDeltaMapping.train(
#         obs, sim, nquantiles=101, kind="*", group="time.month"
# )

# print("QDM trained; dataset summary:")
# print(qdm.ds)

# # Save trained QDM to NetCDF
# print(f"Saving trained QDM to {OUTPUT_QDM_TRAINED} ...")
# with ProgressBar():
#     qdm.ds.to_netcdf(OUTPUT_QDM_TRAINED, compute=True)
# print("QDM saved.")

In [ ]:
# -----------------------------
# Apply QDM to test data
# -----------------------------
print("Applying QDM to test window...")
adjTest = qdm.adjust(
    downTrace, extrapolation="constant", interp="linear"
)

print("Persisting adjusted dataset across workers...")
adjTest = adjTest.persist()

if isinstance(adjTest, xr.DataArray):
    ds_out = adjTest.to_dataset(name="pr")
else:
    ds_out = adjTest.ds if hasattr(adjTest, "ds") else adjTest

# Compression/chunk encoding for NetCDF
encoding = {}
for v, da in ds_out.data_vars.items():
    enc = {"zlib": True, "complevel": 4, "shuffle": True}
    chunks = []
    for dim in da.dims:
        if dim == "time":
            # min between target chunk size and actual size
            chunks.append(min(MONTHS_PER_CHUNK, da.sizes[dim]))
        elif dim == "lat":
            chunks.append(min(lat_chunk, da.sizes[dim]))
        elif dim == "lon":
            chunks.append(min(lon_chunk, da.sizes[dim]))
        else:
            # don’t chunk unexpected dims (e.g., scalar or level)
            continue
    if chunks:
        enc["chunksizes"] = tuple(chunks)
    encoding[v] = enc


print(f"Writing adjusted dataset to {OUTPUT_ADJUSTED} ...")
with ProgressBar():
    ds_out.to_netcdf(
        OUTPUT_ADJUSTED,
        compute=True,
        format="NETCDF4",
        encoding=encoding,
        engine="h5netcdf",  # or "netcdf4"
    )

print("Done.")

In [ ]:
ds = xr.open_dataset(OUTPUT_ADJUSTED)
print(ds)